<a href="https://colab.research.google.com/github/aakriti1318/llm-inference-sandbox/blob/main/vLLM-101/vLLM_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vLLM-101: Local LLM Serving & Inference

A practical notebook for learning how to deploy and interact with LLMs locally:
- **In-Code Inference:** Loading and prompting Hugging Face models directly in Python using vLLM
- **Local API Server:** Launching a background server to handle requests via an OpenAI-compatible interface
- **Custom GGUF Deployment:** Serving quantized custom fine-tuned GGUF weights alongside custom tokenizers and Jinja chat templates.

### Initialization

In [1]:
!uv init

Initialized project `content`


In [2]:
!uv venv --clear

Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [ ]:
# Install vLLM and its automatic torch backend dependencies optimized for the hardware

!uv pip install vllm --torch-backend=auto

In [4]:
import torch
print("Is CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Is CUDA available: True
Device name: Tesla T4


In [7]:
!uv pip uninstall torchaudio

Using Python 3.13.15 environment at: /usr
Uninstalled 1 package in 151ms
 - torchaudio==2.11.0+cu128


## 01 - In-Code Inference
Loading and prompting Hugging Face models directly in Python using vLLM

In [13]:
import sys
from types import ModuleType

# Create a dummy torchaudio module to satisfy transformers' import check
dummy_torchaudio = ModuleType("torchaudio")
dummy_torchaudio.__version__ = "2.6.0"  # fake version string
sys.modules["torchaudio"] = dummy_torchaudio

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-Coder-0.5B-Instruct",
    gpu_memory_utilization=0.2
)

params = SamplingParams(max_tokens=128, temperature=0.7)

prompt = "<|im_start|>user\nWhat is special about Python language?<|im_end|>\n<|im_start|>assistant\n"

outputs = llm.generate(prompt, sampling_params=params)

print(outputs[0].outputs[0].text.strip())

INFO 08-27 20:56:59 [api_utils.py:272] non-default args: {'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct'}
INFO 08-27 20:57:00 [model.py:672] Resolved architecture: Qwen2ForCausalLM
WARNING 08-27 20:57:00 [model.py:2246] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 08-27 20:57:00 [model.py:2299] Casting torch.bfloat16 to torch.float16.
INFO 08-27 20:57:00 [model.py:1965] Using max model len 32768
INFO 08-27 20:57:00 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 08-27 20:57:43 [utils.py:615] [shutdown] Process manager: send sigterm to process EngineCore
WARNING 08-27 20:57:48 [utils.py:635] [shutdown] Process manager: force killing remaining processes count=1
WARNING 08-27 20:57:48 [utils.py:640] [shutdown] Process manager: force killing re

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s, est. speed input: 17.86 toks/s, output: 152.38 toks/s]

Python is a versatile and powerful programming language that is widely used for web development, data analysis, and scripting. Here are some of the special features of Python:

1. **Dynamic Typing**: Python is a dynamically typed language, which means that variables can be assigned values of different types at runtime. This makes it easy to write code that can handle a wide range of data types.

2. **Object-Oriented Programming (OOP)**: Python has an object-oriented programming language that emphasizes the use of classes and objects. It provides a set of built-in functions and methods that can be used to manipulate data and perform operations.

3


In [11]:
# Inspect active GPU memory utilization and process allocation via nvidia-smi

!nvidia-smi

Thu Aug 27 20:55:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             27W /   70W |    3441MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
print(outputs[0].outputs[0].text.strip())

Python is a versatile and powerful programming language that is widely used for web development, data analysis, and scripting. Here are some of the special features of Python:

1. **Dynamic Typing**: Python is a dynamically typed language, which means that variables can be assigned values of different types at runtime. This makes it easy to write code that can handle a wide range of data types.

2. **Object-Oriented Programming (OOP)**: Python has an object-oriented programming language that emphasizes the use of classes and objects. It provides a set of built-in functions and methods that can be used to manipulate data and perform operations.

3


## 02 - Local API Server

The command below launches vLLM as an independent background daemon, hosting an OpenAI-compatible API endpoint with a restricted GPU memory footprint.

In [17]:
# Start the vLLM server in the background and pipe all logs to vllm.log

!nohup vllm serve Qwen/Qwen2.5-Coder-0.5B-Instruct --gpu_memory_utilization 0.2 --api_key aakey > vllm.log 2>&1 &

In [21]:
!cat vllm.log

(APIServer pid=8790) INFO 08-27 21:06:44 [api_utils.py:333] 
(APIServer pid=8790) INFO 08-27 21:06:44 [api_utils.py:333]        █     █     █▄   ▄█
(APIServer pid=8790) INFO 08-27 21:06:44 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=8790) INFO 08-27 21:06:44 [api_utils.py:333]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-Coder-0.5B-Instruct
(APIServer pid=8790) INFO 08-27 21:06:44 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=8790) INFO 08-27 21:06:44 [api_utils.py:333] 
(APIServer pid=8790) INFO 08-27 21:06:44 [api_utils.py:272] non-default args: {'model_tag': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'api_key': ['aakey'], 'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'gpu_memory_utilization': 0.2}
(APIServer pid=8790) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=8790) INFO 08-27 21:06:45 [model.py:672] Resolved architecture: Qwen2For

#### Client Interaction
Once the server is up, standard API client libraries can communicate with the local instance just like a cloud-hosted provider.

In [ ]:
!uv add openai

In [26]:
# Query the locally hosted vLLM server using the standard OpenAI client interface

from openai import OpenAI

client = OpenAI(base_url='http://localhost:8000/v1', api_key="aakey")

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-Coder-0.5B-Instruct",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful expert"
        },
        {
            "role": "user",
            "content": "Explain Python language?"
        }

    ],
)

print(response.choices[0].message.content)

Python is an interpreted high-level programming language designed to be easy to read and write. It's widely used for web development, scientific computing, data analysis, and more. Here's a basic overview of what Python does:

1. **Syntax**: Python uses a syntax similar to that of many other languages, such as C, Java, or JavaScript. It uses indentation to define blocks of code.

2. **Data Types**: Python supports various data types, including integers, floats, strings, lists, tuples, dictionaries, sets, and more. Each data type has specific properties and methods.

3. **Control Structures**: Python includes control structures like `if`, `elif`, `else`, `for`, `while`, and `break`. These structures allow you to perform conditional checks and loops.

4. **Functions**: Functions are defined using the `def` keyword, and they can take parameters and return values. Functions can be called to perform specific tasks.

5. **Modules**: Python allows you to organize your code into modules, which

## 03 - Custom GGUF Model Serving

To deploy a custom quantized GGUF model alongside its dedicated tokenizer configurations and Jinja chat templates, configure the serve command pointing directly to the file paths


```
vllm serve ./ \
  --model ./unsloth.Q4_K_M.gguf \
  --served-model-name custom-finetuned-model \
  --chat-template ./chat_template.jinja
```



In [ ]:
!vllm serve "./unsloth.Q4_K_M.gguf" \
  --tokenizer "scratch_smaller" \
  --served-model-name tunedtutorialmodel \
  --chat-template "scratch_smaller/chat_template.jinja" \
  --gpu-memory-utilization 0.2 \
  --api-key aakey